# 088 — Texto a imagen y condicionamiento

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Texto a imagen** = difusión (clase 087) + tres piezas:

1. **Encoder de texto (CLIP)**: el prompt se convierte en una secuencia de
   embeddings c (en SD 1.x, 77 tokens × 768 dims) con semántica visual aprendida
   por contraste imagen-texto.
2. **Cross-attention**: la U-Net predice ε_θ(x_t, t, c); en sus capas de atención
   cruzada las queries vienen de las features espaciales y las keys/values del
   texto — cada región de la imagen "consulta" qué palabras le importan.
3. **Classifier-free guidance (CFG)**: en inferencia se combinan dos predicciones,
   `ε̃ = ε_θ(x_t, ∅) + w·(ε_θ(x_t, c) − ε_θ(x_t, ∅))`, con w ≈ 7.5. Es una
   **extrapolación** (no interpolación): más adherencia al prompt a costa de
   diversidad, y dos pasadas de U-Net por paso.

**Latent diffusion (Stable Diffusion)**: toda la difusión ocurre en el latente
z = E(x) de un autoencoder (64×64×4 en vez de 512×512×3, ~48× menos valores);
el decoder D(z₀) reconstruye la imagen al final.


## 🧮 Ejemplo de referencia

Con ε_∅ = 0.20, ε_c = 0.32 y w = 7.5:
`ε̃ = 0.20 + 7.5·(0.32 − 0.20) = 1.10` — fuera del intervalo [0.20, 0.32]:
la guía extrapola, por eso w alto satura la imagen.

Ahorro del espacio latente: 512·512·3 = 786 432 valores frente a
64·64·4 = 16 384 → factor 48×. Con 50 pasos y CFG son 100 evaluaciones de
U-Net; hacerlas sobre el latente es lo que vuelve viable la generación.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=88)
show(result)


## Reflexión

1. ¿Por qué CFG exige que el modelo se haya entrenado también con la condición vacía ∅ (dropout del prompt), y qué pasaría si nunca hubiera visto ejemplos incondicionales?
2. Subir w de 7.5 a 20 no da "más obediencia gratis": ¿qué efectos degenerativos aparecen y cómo los explica el hecho de que CFG extrapola fuera del intervalo [ε_∅, ε_c]?
3. Un texto pequeño ilegible dentro de la imagen, ¿es culpa de la U-Net de difusión o del decoder del VAE? Diseña un experimento que lo distinga (pista: reconstruye una imagen real con el autoencoder, sin difusión).
